In [ ]:
from pathlib import Path

import dask as da
from dask.distributed import Client
from dask_jobqueue.lsf import LSFCluster

from comptest.utils.state_guard import run_once_guard

In [2]:
PROJECT_DIR = Path().resolve()

In [3]:
# clear_run_once_guard("cluster")

In [4]:
cluster: LSFCluster
client: da.distributed.Client

if run_once_guard("cluster"):
    da.config.set(
        {
            "distributed.scheduler.worker-ttl": "30s",  # Ensures we don't leave hanging working when restarting notebook etc
        }
    )
    cluster = LSFCluster(
        queue="hpc",
        cores=4,
        memory="8GB",
        walltime="01:00",
        worker_extra_args=[
            "--lifetime",
            "45m",
            "--lifetime-stagger",
            "5m",
        ],
        job_extra_directives=[
            '-R "select[avx512]"',
            f'-e "{str(PROJECT_DIR / "logs" / "dask-worker-%J.%I.err")}"',
            f'-o "{str(PROJECT_DIR / "logs" / "dask-worker-%J.%I.out")}"',
        ],
    )
    client = Client(
        cluster,
    )

    cluster.adapt(minimum=2, maximum=4)

In [5]:
cluster.adapt(minimum_jobs=2, maximum_jobs=4)

In [6]:
client.get_scheduler_logs()

(('INFO',
  '2026-07-29 11:48:20,596 - distributed.scheduler - INFO - Receive client connection: Client-a1e0cbfb-8b32-11f1-91eb-d4f5ef394e60'),
 ('INFO',
  '2026-07-29 11:48:20,564 - distributed.scheduler - INFO - Registering Worker plugin shuffle'),
 ('INFO',
  '2026-07-29 11:48:20,564 - distributed.scheduler - INFO -   dashboard at:  http://10.66.12.19:8787/status'),
 ('INFO',
  '2026-07-29 11:48:20,564 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.66.12.19:35031'),
 ('INFO',
  '2026-07-29 11:48:20,560 - distributed.scheduler - INFO - State start'))

In [7]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.LSFCluster
Dashboard: http://10.66.12.19:8787/status,
Dashboard: http://10.66.12.19:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.66.12.19:35031,Workers: 0
Dashboard: http://10.66.12.19:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [ ]:
cluster

Dashboard: http://10.66.12.19:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.66.12.19:35031,Workers: 0
Dashboard: http://10.66.12.19:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


2026-07-29 12:00:19,205 - tornado.application - ERROR - Exception in callback <function Cluster._widget.<locals>.update at 0x7ff58c07fed0>
Traceback (most recent call last):
  File "/zhome/1a/a/219376/Code/qim-naturarv3d-compression/.pixi/envs/default/lib/python3.14/site-packages/tornado/ioloop.py", line 945, in _run
  File "/zhome/1a/a/219376/Code/qim-naturarv3d-compression/.pixi/envs/default/lib/python3.14/site-packages/distributed/deploy/cluster.py", line 461, in update
  File "/zhome/1a/a/219376/Code/qim-naturarv3d-compression/.pixi/envs/default/lib/python3.14/site-packages/distributed/deploy/cluster.py", line 478, in _repr_html_
  File "/zhome/1a/a/219376/Code/qim-naturarv3d-compression/.pixi/envs/default/lib/python3.14/site-packages/distributed/objects.py", line 44, in _repr_html_
  File "/zhome/1a/a/219376/Code/qim-naturarv3d-compression/.pixi/envs/default/lib/python3.14/site-packages/jinja2/environment.py", line 1016, in get_template
  File "/zhome/1a/a/219376/Code/qim-naturarv

In [9]:
client.get_scheduler_logs()

(('INFO',
  '2026-07-29 11:48:20,596 - distributed.scheduler - INFO - Receive client connection: Client-a1e0cbfb-8b32-11f1-91eb-d4f5ef394e60'),
 ('INFO',
  '2026-07-29 11:48:20,564 - distributed.scheduler - INFO - Registering Worker plugin shuffle'),
 ('INFO',
  '2026-07-29 11:48:20,564 - distributed.scheduler - INFO -   dashboard at:  http://10.66.12.19:8787/status'),
 ('INFO',
  '2026-07-29 11:48:20,564 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.66.12.19:35031'),
 ('INFO',
  '2026-07-29 11:48:20,560 - distributed.scheduler - INFO - State start'))

In [10]:
import socket

socket.gethostname()


def get_hostname() -> str:
    return socket.gethostname()


print(da.delayed(get_hostname)().compute())
print(get_hostname())

n-62-30-24
n-62-12-19
